# RAG Pipeline on OpenShift AI — Setup, Run & Query Guide

This notebook walks you through the complete **Retrieval-Augmented Generation (RAG)** pipeline on **Red Hat OpenShift AI (RHOAI)**.

## Pipeline Overview

The pipeline has **4 steps**, orchestrated by Kubeflow Pipelines (KFP):

| Step | Component | Description |
|------|-----------|-------------|
| 1 | **Parse & Chunk** | RayJob distributes PDF parsing via Docling + HybridChunker, writes JSONL to S3 (MinIO) |
| 2 | **Ingest to Milvus** | Reads chunks from S3, embeds with local or deployed model, inserts into Milvus |
| 3 | **Download Model** | Downloads LLM weights to a PVC (cached — skips if already present) |
| 4 | **Deploy Model** | Creates a vLLM InferenceService from the cached PVC model |

```
  Input PDFs (PVC)              HuggingFace
       │                             │
       ▼                             ▼
  ┌──────────┐                ┌──────────────┐
  │ 1. Parse │                │ 3. Download  │
  │ & Chunk  │                │    Model     │
  │ (RayJob) │                │  (to PVC)    │
  └────┬─────┘                └──────┬───────┘
       │ JSONL → S3                  │
       ▼                             ▼
  ┌──────────┐                ┌──────────────┐
  │ 2. Ingest│                │ 4. Deploy    │
  │ to Milvus│                │    Model     │
  │ (embed)  │                │  (vLLM ISVC) │
  └────┬─────┘                └──────┬───────┘
       │                             │
       ▼                             ▼
    Milvus DB ◄──── RAG Query ────► vLLM API
```

## Embedding Modes

This pipeline supports **two modes** for generating embeddings:

**Mode 1: Local** (default)
- Embedding model runs inside the pipeline component
- Uses `sentence-transformers` library
- Simpler setup, good for testing
- Model loaded for each pipeline run

**Mode 2: Service**
- Embedding model deployed as a dedicated InferenceService
- Reusable across multiple runs and queries
- Better resource utilization
- Can leverage GPU acceleration
- Ideal for production

Toggle between modes by setting `EMBEDDING_MODE = "local"` or `"service"` in the configuration.

## What This Notebook Covers

1. **Prerequisites** — Create PVCs, Secrets, RBAC
2. **Deploy Embedding Service** (optional - for service mode)
3. **Compile the Pipeline** — Generate the KFP YAML
4. **Upload & Run** — Submit to Data Science Pipelines
5. **Monitor** — Track pipeline and component progress
6. **Test RAG** — Query the deployed model using Milvus-retrieved context

---
## 1. Prerequisites

Before running the pipeline, ensure the following are in place:

- **RHOAI** installed with Data Science Pipelines (DSPA) configured
- **KubeRay** operator enabled
- **Milvus** deployed (e.g., via Milvus Operator)
- **MinIO** or S3-compatible storage for intermediate chunks
- **GPU nodes** with NVIDIA GPUs for model serving

In [ ]:
# Install required packages
!pip install -q kfp kfp-kubernetes pymilvus sentence-transformers openai requests kubernetes codeflare-sdk pyyaml tabulate

In [ ]:
# ============================================================
# Configuration — Update these values for your environment
# ============================================================

NAMESPACE = "ray-docling"

# S3 / MinIO
S3_ENDPOINT = "http://minio-service.default.svc.cluster.local:9000"
S3_BUCKET = "rag-chunks"
S3_PREFIX = "chunks"
S3_SECRET_NAME = "minio-secret"  # K8s Secret with keys: access_key, secret_key

# PDF Input
# IMPORTANT: Update this to match where your PDFs are located on the data-pvc
INPUT_PATH = "input/pdfs"  # ← UPDATE THIS: path relative to pvc_mount_path

# Milvus
MILVUS_HOST = "milvus-milvus.milvus.svc.cluster.local"
MILVUS_PORT = 19530
COLLECTION_NAME = "paper_rag_documents"  # Update to match your collection name

# Embedding Configuration
# MODE 1: Local embedding (SentenceTransformer in pipeline)
# MODE 2: Deployed embedding service (InferenceService in RHOAI)
EMBEDDING_MODE = "local"  # Options: "local" or "service"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_DIM = 384
# Only used when EMBEDDING_MODE = "service"
EMBEDDING_SERVICE_NAME = "embedding-service"
EMBEDDING_ENDPOINT = f"http://{EMBEDDING_SERVICE_NAME}.{NAMESPACE}.svc.cluster.local:8080"

# LLM
LLM_MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
LLM_SERVED_NAME = "mistral-7b-instruct-v0-3"  # As registered in vLLM
MODEL_CACHE_PVC = "model-cache-pvc"

# Inference endpoint
# NOTE: The default InferenceService predictor service is headless (ClusterIP: None).
# We created a proper ClusterIP service for in-cluster access.
INFERENCE_URL = "http://mistral-7b-instruct-internal.ray-docling.svc.cluster.local:8080/v1"

print(f"Namespace:      {NAMESPACE}")
print(f"S3 endpoint:    {S3_ENDPOINT}")
print(f"Milvus:         {MILVUS_HOST}:{MILVUS_PORT}")
print(f"Collection:     {COLLECTION_NAME}")
print(f"Input path:     {INPUT_PATH}")
print(f"Embedding mode: {EMBEDDING_MODE}")
if EMBEDDING_MODE == "service":
    print(f"Embedding URL:  {EMBEDDING_ENDPOINT}")
else:
    print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"LLM endpoint:   {INFERENCE_URL}")

### Verify LLM Service

The pipeline creates an InferenceService, but the default predictor service is headless.
We need a proper ClusterIP service for in-cluster access from this notebook.

In [ ]:
# Check if the internal service exists, create if not
from kubernetes import client, config
from kubernetes.client.rest import ApiException
import yaml

# Load kubeconfig
try:
    config.load_incluster_config()
except:
    config.load_kube_config()

v1 = client.CoreV1Api()
service_name = "mistral-7b-instruct-internal"

try:
    service = v1.read_namespaced_service(name=service_name, namespace=NAMESPACE)
    print(f"✅ Service '{service_name}' exists")
    print(f"   ClusterIP: {service.spec.cluster_ip}")
except ApiException as e:
    if e.status == 404:
        print(f"⚠ Service '{service_name}' not found. Creating it...")
        
        # Load and apply the service manifest
        with open("../manifests/llm-inference-service.yaml", "r") as f:
            service_manifest = yaml.safe_load(f)
        
        try:
            v1.create_namespaced_service(namespace=NAMESPACE, body=service_manifest)
            print(f"✅ Service '{service_name}' created successfully")
        except ApiException as create_error:
            print(f"❌ Failed to create service: {create_error}")
            print("You can create it manually:")
            print(f"  kubectl apply -f ../manifests/llm-inference-service.yaml")
    else:
        print(f"❌ Error checking service: {e}")

### 1.1 Create Prerequisites (run once)

Apply the required Kubernetes manifests. Skip if already created.

In [ ]:
# Create the S3 credentials Secret
from kubernetes import client
from kubernetes.client.rest import ApiException

v1 = client.CoreV1Api()

secret = client.V1Secret(
    metadata=client.V1ObjectMeta(name="minio-secret", namespace=NAMESPACE),
    string_data={
        "access_key": "minioadmin",
        "secret_key": "minioadmin"
    },
    type="Opaque"
)

try:
    v1.create_namespaced_secret(namespace=NAMESPACE, body=secret)
    print(f"✅ Secret 'minio-secret' created in namespace '{NAMESPACE}'")
except ApiException as e:
    if e.status == 409:  # Already exists
        # Patch/update the secret
        v1.patch_namespaced_secret(name="minio-secret", namespace=NAMESPACE, body=secret)
        print(f"✅ Secret 'minio-secret' updated in namespace '{NAMESPACE}'")
    else:
        print(f"❌ Error creating secret: {e}")

In [ ]:
# Create the model cache PVC (30Gi for Mistral-7B weights)
from kubernetes import client
from kubernetes.client.rest import ApiException
import yaml

v1 = client.CoreV1Api()

# Load the PVC manifest
with open("manifests/model-cache-pvc.yaml", "r") as f:
    pvc_manifest = yaml.safe_load(f)

try:
    v1.create_namespaced_persistent_volume_claim(namespace=NAMESPACE, body=pvc_manifest)
    print(f"✅ PVC '{pvc_manifest['metadata']['name']}' created")
except ApiException as e:
    if e.status == 409:
        print(f"✅ PVC '{pvc_manifest['metadata']['name']}' already exists")
    else:
        print(f"❌ Error creating PVC: {e}")

In [ ]:
# Create RBAC for the pipeline service account
from kubernetes import client
from kubernetes.client.rest import ApiException
import yaml

rbac_v1 = client.RbacAuthorizationV1Api()

# Load the RBAC manifest
with open("manifests/rbac.yaml", "r") as f:
    rbac_docs = yaml.safe_load_all(f)
    
    for doc in rbac_docs:
        if not doc:
            continue
            
        kind = doc.get("kind")
        
        try:
            if kind == "Role":
                rbac_v1.create_namespaced_role(namespace=NAMESPACE, body=doc)
                print(f"✅ Role '{doc['metadata']['name']}' created")
            elif kind == "RoleBinding":
                rbac_v1.create_namespaced_role_binding(namespace=NAMESPACE, body=doc)
                print(f"✅ RoleBinding '{doc['metadata']['name']}' created")
        except ApiException as e:
            if e.status == 409:
                print(f"✅ {kind} '{doc['metadata']['name']}' already exists")
            else:
                print(f"❌ Error creating {kind}: {e}")

In [ ]:
# Verify all prerequisites
from kubernetes import client
from tabulate import tabulate

v1 = client.CoreV1Api()
rbac_v1 = client.RbacAuthorizationV1Api()

print("=== PVCs ===")
pvcs = v1.list_namespaced_persistent_volume_claim(namespace=NAMESPACE)
pvc_data = [[pvc.metadata.name, pvc.status.phase, pvc.spec.resources.requests.get('storage')] 
            for pvc in pvcs.items]
print(tabulate(pvc_data, headers=["NAME", "STATUS", "CAPACITY"], tablefmt="simple"))

print("\n=== Secrets ===")
try:
    secret = v1.read_namespaced_secret(name=S3_SECRET_NAME, namespace=NAMESPACE)
    print(f"✅ {S3_SECRET_NAME} exists with {len(secret.data)} keys")
except ApiException as e:
    print(f"❌ Secret {S3_SECRET_NAME} not found: {e}")

print("\n=== RBAC ===")
try:
    role = rbac_v1.read_namespaced_role(name="pipeline-rag-role", namespace=NAMESPACE)
    print(f"✅ Role 'pipeline-rag-role' exists with {len(role.rules)} rules")
except ApiException as e:
    print(f"❌ Role not found: {e}")

print("\n=== Pipeline Service Account ===")
sas = v1.list_namespaced_service_account(namespace=NAMESPACE)
pipeline_sas = [sa.metadata.name for sa in sas.items if 'pipeline' in sa.metadata.name or 'dspa' in sa.metadata.name]
for sa in pipeline_sas:
    print(f"  - {sa}")

In [ ]:
# Test embedding service endpoint (only if EMBEDDING_MODE == "service")
if EMBEDDING_MODE == "service":
    import requests
    import time
    
    print(f"Testing embedding endpoint: {EMBEDDING_ENDPOINT}")
    
    # Wait for service to be ready
    max_retries = 30
    retry_interval = 5
    
    for i in range(max_retries):
        try:
            resp = requests.post(
                f"{EMBEDDING_ENDPOINT}/v1/embeddings",
                json={
                    "model": EMBEDDING_MODEL,
                    "input": ["test sentence"]
                },
                timeout=10
            )
            resp.raise_for_status()
            result = resp.json()
            
            embedding = result['data'][0]['embedding']
            print(f"\n✅ Embedding service is ready!")
            print(f"   Model: {EMBEDDING_MODEL}")
            print(f"   Embedding dimension: {len(embedding)}")
            print(f"   Sample: {embedding[:5]}...")
            break
            
        except requests.ConnectionError:
            if i < max_retries - 1:
                print(f"⏳ Service not ready yet, retrying in {retry_interval}s... ({i+1}/{max_retries})")
                time.sleep(retry_interval)
            else:
                print(f"\n❌ Service did not become ready after {max_retries * retry_interval}s")
                print("   Check the pod logs for errors")
        except Exception as e:
            print(f"❌ Error testing endpoint: {e}")
            break
else:
    print(f"Skipping embedding service test (EMBEDDING_MODE = '{EMBEDDING_MODE}')")

In [ ]:
# Check embedding service status (only if EMBEDDING_MODE == "service")
if EMBEDDING_MODE == "service":
    from kubernetes import client
    from tabulate import tabulate
    import time
    
    custom_api = client.CustomObjectsApi()
    v1 = client.CoreV1Api()
    
    print("Checking embedding service deployment status...")
    print("\n=== InferenceService ===")
    
    try:
        isvc = custom_api.get_namespaced_custom_object(
            group="serving.kserve.io",
            version="v1beta1",
            namespace=NAMESPACE,
            plural="inferenceservices",
            name=EMBEDDING_SERVICE_NAME
        )
        
        status = isvc.get('status', {})
        conditions = status.get('conditions', [])
        
        if conditions:
            for cond in conditions:
                print(f"  {cond['type']}: {cond['status']}")
                if cond.get('message'):
                    print(f"    Message: {cond['message']}")
        
        url = status.get('url', 'N/A')
        print(f"\n  URL: {url}")
        
    except ApiException as e:
        print(f"❌ Error getting InferenceService: {e}")
    
    print("\n=== Embedding Service Pod ===")
    pods = v1.list_namespaced_pod(
        namespace=NAMESPACE,
        label_selector=f"serving.kserve.io/inferenceservice={EMBEDDING_SERVICE_NAME}"
    )
    
    if pods.items:
        pod_data = []
        for pod in pods.items:
            name = pod.metadata.name
            status = pod.status.phase
            ready = sum(1 for c in pod.status.container_statuses if c.ready) if pod.status.container_statuses else 0
            total = len(pod.status.container_statuses) if pod.status.container_statuses else 0
            pod_data.append([name, status, f"{ready}/{total}"])
        print(tabulate(pod_data, headers=["NAME", "STATUS", "READY"], tablefmt="simple"))
    else:
        print(f"⏳ No pods found yet (may still be starting)")
        print(f"   Run this cell again in a few moments")
else:
    print(f"Skipping embedding service check (EMBEDDING_MODE = '{EMBEDDING_MODE}')")

In [ ]:
# Deploy embedding model as InferenceService (only if EMBEDDING_MODE == "service")
from kubernetes import client
from kubernetes.client.rest import ApiException

if EMBEDDING_MODE == "service":
    custom_api = client.CustomObjectsApi()
    
    # Create ServingRuntime for text-embeddings-inference (TEI)
    serving_runtime = {
        "apiVersion": "serving.kserve.io/v1alpha1",
        "kind": "ServingRuntime",
        "metadata": {
            "name": EMBEDDING_SERVICE_NAME,
            "namespace": NAMESPACE
        },
        "spec": {
            "containers": [{
                "name": "kserve-container",
                "image": "ghcr.io/huggingface/text-embeddings-inference:latest",
                "args": [
                    "--model-id", EMBEDDING_MODEL,
                    "--port", "8080",
                    "--max-batch-tokens", "8192"
                ],
                "ports": [{
                    "containerPort": 8080,
                    "protocol": "TCP"
                }],
                "resources": {
                    "requests": {
                        "cpu": "2",
                        "memory": "4Gi"
                    },
                    "limits": {
                        "cpu": "4",
                        "memory": "8Gi"
                    }
                }
            }],
            "supportedModelFormats": [{
                "name": "huggingface",
                "version": "1"
            }]
        }
    }
    
    # Create InferenceService
    inference_service = {
        "apiVersion": "serving.kserve.io/v1beta1",
        "kind": "InferenceService",
        "metadata": {
            "name": EMBEDDING_SERVICE_NAME,
            "namespace": NAMESPACE,
            "annotations": {
                "serving.kserve.io/deploymentMode": "RawDeployment"
            }
        },
        "spec": {
            "predictor": {
                "model": {
                    "modelFormat": {
                        "name": "huggingface"
                    },
                    "runtime": EMBEDDING_SERVICE_NAME,
                    "resources": {
                        "requests": {
                            "cpu": "2",
                            "memory": "4Gi"
                        },
                        "limits": {
                            "cpu": "4",
                            "memory": "8Gi"
                        }
                    }
                }
            }
        }
    }
    
    # Apply ServingRuntime
    try:
        custom_api.create_namespaced_custom_object(
            group="serving.kserve.io",
            version="v1alpha1",
            namespace=NAMESPACE,
            plural="servingruntimes",
            body=serving_runtime
        )
        print(f"✅ ServingRuntime '{EMBEDDING_SERVICE_NAME}' created")
    except ApiException as e:
        if e.status == 409:
            print(f"✅ ServingRuntime '{EMBEDDING_SERVICE_NAME}' already exists")
        else:
            print(f"❌ Error creating ServingRuntime: {e}")
    
    # Apply InferenceService
    try:
        custom_api.create_namespaced_custom_object(
            group="serving.kserve.io",
            version="v1beta1",
            namespace=NAMESPACE,
            plural="inferenceservices",
            body=inference_service
        )
        print(f"✅ InferenceService '{EMBEDDING_SERVICE_NAME}' created")
        print(f"   Endpoint: {EMBEDDING_ENDPOINT}")
    except ApiException as e:
        if e.status == 409:
            print(f"✅ InferenceService '{EMBEDDING_SERVICE_NAME}' already exists")
        else:
            print(f"❌ Error creating InferenceService: {e}")
else:
    print(f"Skipping embedding service deployment (EMBEDDING_MODE = '{EMBEDDING_MODE}')")

---
## 1.2 Deploy Embedding Model (Optional - for "service" mode)

If you set `EMBEDDING_MODE = "service"`, deploy the embedding model as an InferenceService.
This allows the embedding model to be shared across multiple pipeline runs and RAG queries.

**Benefits of service mode:**
- Reusable across multiple pipeline runs
- Can leverage GPU acceleration
- Centralized embedding endpoint for consistency
- Better resource utilization

**Skip this section if using `EMBEDDING_MODE = "local"`**

---
## 2. Compile the Pipeline

The pipeline is defined in `pipeline_multistep.py` with 4 reusable KFP components.
Compiling generates a self-contained YAML that can be uploaded to Data Science Pipelines.

In [ ]:
!python pipeline_multistep.py

In [ ]:
import os
yaml_path = "rag_multistep_pipeline.yaml"
size_kb = os.path.getsize(yaml_path) / 1024
print(f"Compiled: {yaml_path} ({size_kb:.1f} KB)")

---
## 3. Upload & Run the Pipeline

You can upload the pipeline YAML to Data Science Pipelines via:

**Option A — KFP SDK (programmatic):**

This approach lets you customize all pipeline parameters programmatically.

In [ ]:
# Upload and run via KFP SDK
import kfp
from kubernetes import client
import subprocess

# Get the DSPA route using Kubernetes client
custom_api = client.CustomObjectsApi()

try:
    route = custom_api.get_namespaced_custom_object(
        group="route.openshift.io",
        version="v1",
        namespace=NAMESPACE,
        plural="routes",
        name="ds-pipeline-dspa"
    )
    dspa_host = route['spec']['host']
    dspa_url = f"https://{dspa_host}"
    print(f"DSPA URL: {dspa_url}")
except Exception as e:
    print(f"Error getting route: {e}")
    print("Falling back to CLI method...")
    dspa_route = !oc get route ds-pipeline-dspa -n {NAMESPACE} -o jsonpath='{{.spec.host}}'
    dspa_url = f"https://{dspa_route[0]}"

# Get auth token
token = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True).stdout.strip()

# Create KFP client
kfp_client = kfp.Client(
    host=dspa_url,
    existing_token=token,
    verify_ssl=False,  # Required for self-signed OpenShift route certificates
)

# Upload pipeline
pipeline = kfp_client.upload_pipeline(
    pipeline_package_path="rag_multistep_pipeline.yaml",
    pipeline_name="rag-multi-step-pipeline",  # Must be lowercase with hyphens only
)
print(f"Pipeline uploaded: {pipeline.pipeline_id}")

# Determine embedding endpoint based on mode
embedding_endpoint_param = EMBEDDING_ENDPOINT if EMBEDDING_MODE == "service" else ""

# Create a run with all parameters (modify as needed)
run = kfp_client.create_run_from_pipeline_package(
    pipeline_file="rag_multistep_pipeline.yaml",
    run_name="rag-test-run",
    arguments={
        # Shared
        "pvc_name": "data-pvc",
        "pvc_mount_path": "/mnt/data",
        "namespace": NAMESPACE,
        
        # S3 (MinIO)
        "s3_endpoint": S3_ENDPOINT,
        "s3_bucket": S3_BUCKET,
        "s3_prefix": S3_PREFIX,
        "s3_secret_name": S3_SECRET_NAME,
        
        # PDF parsing
        "input_path": INPUT_PATH,
        "ray_image": "quay.io/rhoai-szaher/docling-ray:latest",
        "num_workers": 2,
        "worker_cpus": 8,
        "worker_memory_gb": 16,
        "cpus_per_actor": 4,
        "min_actors": 2,
        "max_actors": 4,
        "chunk_max_tokens": 256,
        "num_files": 1000,
        
        # Embedding - supports both local and service modes
        "embedding_endpoint": embedding_endpoint_param,
        "embedding_model": EMBEDDING_MODEL,
        "embedding_dim": EMBEDDING_DIM,
        
        # Milvus
        "milvus_host": MILVUS_HOST,
        "milvus_port": MILVUS_PORT,
        "collection_name": COLLECTION_NAME,
        
        # LLM deployment
        "llm_model_name": LLM_MODEL_NAME,
        "model_cache_pvc": MODEL_CACHE_PVC,
        "max_model_len": 4096,
        "gpu_count": 1,
    },
)
print(f"Run started: {run.run_id}")
print(f"\nPipeline configuration:")
print(f"  PDFs from: /mnt/data/{INPUT_PATH}")
print(f"  Embedding mode: {EMBEDDING_MODE}")
if EMBEDDING_MODE == "service":
    print(f"  Embedding endpoint: {embedding_endpoint_param}")
else:
    print(f"  Embedding model: {EMBEDDING_MODEL} (local)")

**Note:** You'll see `InsecureRequestWarning` messages when running the cell above. These are expected when using `verify_ssl=False` with self-signed OpenShift route certificates and are safe to ignore in development/testing environments. For production deployments, see `docs/ssl-fix-options.md` for proper SSL certificate configuration.

**Option B — RHOAI Dashboard (alternative):**

1. Open the RHOAI dashboard
2. Go to **Data Science Pipelines** → **Pipelines**
3. Click **Import pipeline** → upload `rag_multistep_pipeline.yaml`
4. Click **Create run**, review the parameters, and start

Note: The SDK approach above gives you more control over parameters.

---
## 4. Monitor Pipeline Execution

Use these cells to check the status of each pipeline step.

In [ ]:
# Step 1: Check RayJob status (Parse & Chunk)
from kubernetes import client
from tabulate import tabulate

custom_api = client.CustomObjectsApi()
v1 = client.CoreV1Api()

print("=== RayJobs ===")
try:
    rayjobs = custom_api.list_namespaced_custom_object(
        group="ray.io",
        version="v1",
        namespace=NAMESPACE,
        plural="rayjobs"
    )
    
    if rayjobs['items']:
        job_data = []
        for job in rayjobs['items']:
            name = job['metadata']['name']
            status = job.get('status', {})
            job_status = status.get('jobStatus', 'Unknown')
            job_deployment_status = status.get('jobDeploymentStatus', 'Unknown')
            job_data.append([name, job_status, job_deployment_status])
        print(tabulate(job_data, headers=["NAME", "JOB STATUS", "DEPLOYMENT STATUS"], tablefmt="simple"))
    else:
        print("No RayJobs found")
except Exception as e:
    print(f"Error listing RayJobs: {e}")

print("\n=== Ray Pods ===")
pods = v1.list_namespaced_pod(namespace=NAMESPACE, label_selector="ray.io/node-type")
if pods.items:
    pod_data = [[pod.metadata.name, pod.status.phase, pod.metadata.labels.get('ray.io/node-type')] 
                for pod in pods.items]
    print(tabulate(pod_data, headers=["NAME", "STATUS", "NODE TYPE"], tablefmt="simple"))
else:
    print("No Ray pods found")

In [ ]:
# Step 2: Check pipeline pod logs (Ingest to Milvus)
from kubernetes import client
from tabulate import tabulate
from datetime import datetime

v1 = client.CoreV1Api()

pods = v1.list_namespaced_pod(
    namespace=NAMESPACE,
    label_selector="pipelines.kubeflow.org/v2_component=true"
)

if pods.items:
    # Sort by creation timestamp
    sorted_pods = sorted(pods.items, key=lambda p: p.metadata.creation_timestamp)
    
    pod_data = []
    for pod in sorted_pods:
        name = pod.metadata.name
        status = pod.status.phase
        created = pod.metadata.creation_timestamp.strftime("%Y-%m-%d %H:%M:%S")
        pod_data.append([name, status, created])
    
    print(tabulate(pod_data, headers=["NAME", "STATUS", "CREATED"], tablefmt="simple"))
else:
    print("No pipeline component pods found")

In [ ]:
# Steps 3 & 4: Check model deployment
from kubernetes import client
from tabulate import tabulate

custom_api = client.CustomObjectsApi()
v1 = client.CoreV1Api()

print("=== InferenceService ===")
try:
    isvc_list = custom_api.list_namespaced_custom_object(
        group="serving.kserve.io",
        version="v1beta1",
        namespace=NAMESPACE,
        plural="inferenceservices"
    )
    
    if isvc_list['items']:
        isvc_data = []
        for isvc in isvc_list['items']:
            name = isvc['metadata']['name']
            status = isvc.get('status', {})
            ready = status.get('conditions', [{}])[0].get('status', 'Unknown') if status.get('conditions') else 'Unknown'
            url = status.get('url', 'N/A')
            isvc_data.append([name, ready, url])
        print(tabulate(isvc_data, headers=["NAME", "READY", "URL"], tablefmt="simple"))
    else:
        print("No InferenceServices found")
except Exception as e:
    print(f"Error listing InferenceServices: {e}")

print("\n=== Model Serving Pod ===")
pods = v1.list_namespaced_pod(
    namespace=NAMESPACE,
    label_selector=f"serving.kserve.io/inferenceservice={LLM_SERVED_NAME}"
)

if pods.items:
    pod_data = []
    for pod in pods.items:
        name = pod.metadata.name
        status = pod.status.phase
        ready = sum(1 for c in pod.status.container_statuses if c.ready) if pod.status.container_statuses else 0
        total = len(pod.status.container_statuses) if pod.status.container_statuses else 0
        pod_data.append([name, status, f"{ready}/{total}"])
    print(tabulate(pod_data, headers=["NAME", "STATUS", "READY"], tablefmt="simple"))
else:
    print(f"No pods found for InferenceService '{LLM_SERVED_NAME}'")

---
## 5. Test the RAG Pipeline

Once all 4 pipeline steps complete, the RAG system is ready to query:
- **Milvus** contains embedded document chunks
- **vLLM** serves the Mistral-7B model

The RAG flow:
1. Embed the user's question using the same embedding model
2. Search Milvus for the most similar document chunks
3. Build a prompt with the retrieved context
4. Send to the LLM for answer generation

### 5.1 Verify Milvus Collection

Check that the ingestion step populated the vector database.

In [ ]:
from pymilvus import MilvusClient, connections, Collection

milvus_uri = f"http://{MILVUS_HOST}:{MILVUS_PORT}"
print(f"Connecting to {milvus_uri}, Collection:{COLLECTION_NAME}")

# MilvusClient for simple operations
milvus_client = MilvusClient(uri=milvus_uri, db_name="default")

# ORM connection for flush/num_entities (required by Collection API)
connections.connect(alias="default", host=MILVUS_HOST, port=str(MILVUS_PORT))

if not milvus_client.has_collection(COLLECTION_NAME):
    print(f"Collection '{COLLECTION_NAME}' not found.")
    print("Available collections:", milvus_client.list_collections())
    print("The ingestion step may not have completed yet.")
else:
    collection = Collection(COLLECTION_NAME)
    collection.flush()
    milvus_client.load_collection(COLLECTION_NAME)
    print(f"Collection: {COLLECTION_NAME}")
    print(f"Vectors:    {collection.num_entities}")

In [ ]:
# Sample a few chunks to see what's in the collection
samples = milvus_client.query(
    collection_name=COLLECTION_NAME,
    filter="chunk_index >= 0",
    output_fields=["source_file", "chunk_index", "text"],
    limit=5,
)

print(f"Sample chunks ({len(samples)}):\n")
for s in samples:
    print(f"  [{s['source_file']} | chunk {s['chunk_index']}]")
    print(f"  {s['text'][:200]}...")
    print()

### 5.2 Verify Model Endpoint

Check that the vLLM model is serving and responding.

In [ ]:
import requests

print(f"Checking model endpoint: {INFERENCE_URL}")

try:
    resp = requests.get(f"{INFERENCE_URL}/models", timeout=10)
    resp.raise_for_status()
    models = resp.json()
    print(f"\nAvailable models:")
    for m in models.get("data", []):
        print(f"  - {m['id']}")
except requests.ConnectionError:
    print("\nConnection failed. The model may not be deployed yet.")
    print("Check: oc get inferenceservice -n", NAMESPACE)
except Exception as e:
    print(f"\nError: {e}")

### 5.3 RAG Query — Search & Ask

This is the core RAG loop:
1. **Embed** the question with the same model used during ingestion
2. **Search** Milvus for the top-K most similar chunks
3. **Build** a prompt with the retrieved context
4. **Generate** an answer using the LLM

In [ ]:
from openai import OpenAI
import requests

# Initialize embedding client based on mode
embed_model = None
embedding_service_url = None

if EMBEDDING_MODE == "local":
    from sentence_transformers import SentenceTransformer
    embed_model = SentenceTransformer(EMBEDDING_MODEL)
    print(f"Embedding mode: local")
    print(f"  Model: {EMBEDDING_MODEL} (dim={EMBEDDING_DIM})")
else:
    embedding_service_url = EMBEDDING_ENDPOINT
    print(f"Embedding mode: service")
    print(f"  Endpoint: {embedding_service_url}")
    
    # Test the service
    try:
        resp = requests.post(
            f"{embedding_service_url}/v1/embeddings",
            json={"model": EMBEDDING_MODEL, "input": ["test"]},
            timeout=10
        )
        resp.raise_for_status()
        print(f"  ✅ Service is reachable")
    except Exception as e:
        print(f"  ⚠️ Warning: Service not reachable: {e}")

# OpenAI-compatible client pointing to vLLM
llm_client = OpenAI(base_url=INFERENCE_URL, api_key="unused")
print(f"\nLLM client configured: {INFERENCE_URL}")

In [ ]:
def get_embedding(text):
    """Generate embedding for a single text using configured mode."""
    if EMBEDDING_MODE == "local":
        # Use local SentenceTransformer
        return embed_model.encode([text], normalize_embeddings=True).tolist()[0]
    else:
        # Use deployed embedding service
        resp = requests.post(
            f"{embedding_service_url}/v1/embeddings",
            json={"model": EMBEDDING_MODEL, "input": [text]},
            timeout=30
        )
        resp.raise_for_status()
        return resp.json()["data"][0]["embedding"]


def search_similar_chunks(question, top_k=5):
    """Embed the question and search Milvus for similar chunks."""
    query_embedding = get_embedding(question)

    results = milvus_client.search(
        collection_name=COLLECTION_NAME,
        data=[query_embedding],
        limit=top_k,
        output_fields=["source_file", "chunk_index", "text"],
        search_params={"metric_type": "COSINE", "params": {"nprobe": 16}},
    )

    chunks = []
    for hits in results:
        for hit in hits:
            chunks.append({
                "text": hit["entity"]["text"],
                "source_file": hit["entity"]["source_file"],
                "chunk_index": hit["entity"]["chunk_index"],
                "score": hit["distance"],
            })
    return chunks


def build_rag_prompt(question, chunks):
    """Build a prompt with retrieved context for the LLM."""
    context_block = "\n\n---\n\n".join(
        f"[Source: {c['source_file']}, chunk {c['chunk_index']}, "
        f"similarity: {c['score']:.3f}]\n{c['text']}"
        for c in chunks
    )
    return (
        "You are a helpful assistant. Answer the user's question based on the "
        "provided context documents. If the answer is not in the context, say so.\n\n"
        f"## Context\n\n{context_block}\n\n"
        f"## Question\n\n{question}\n\n"
        "## Answer\n\n"
    )


def ask_llm(prompt, max_tokens=512, temperature=0.1):
    """Send the prompt to vLLM and return the response."""
    response = llm_client.chat.completions.create(
        model=LLM_SERVED_NAME,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=temperature,
    )
    return response.choices[0].message.content


def rag_query(question, top_k=5, max_tokens=512, temperature=0.1):
    """End-to-end RAG: retrieve context → build prompt → generate answer."""
    # Step 1: Retrieve similar chunks
    chunks = search_similar_chunks(question, top_k=top_k)

    # Step 2: Build prompt with context
    prompt = build_rag_prompt(question, chunks)

    # Step 3: Generate answer
    answer = ask_llm(prompt, max_tokens=max_tokens, temperature=temperature)

    return {
        "question": question,
        "answer": answer,
        "sources": chunks,
        "prompt": prompt,
    }


print("RAG functions ready.")
print(f"Embedding mode: {EMBEDDING_MODE}")

### 5.4 Try a RAG Query

Ask a question about your ingested documents. The system will:
1. Find the most relevant chunks in Milvus
2. Send them as context to Mistral-7B
3. Return a grounded answer with source references

In [ ]:
question = "What are the main topics covered in the documents?"

result = rag_query(question)

print(f"Question: {result['question']}\n")
print(f"Answer:\n{result['answer']}\n")
print(f"Sources ({len(result['sources'])}):\n")
for s in result["sources"]:
    print(f"  - {s['source_file']} (chunk {s['chunk_index']}, similarity: {s['score']:.3f})")

In [ ]:
# Try another question — replace with something relevant to your documents
result = rag_query("Summarize the key findings.")

print(f"Question: {result['question']}\n")
print(f"Answer:\n{result['answer']}\n")
print(f"Sources ({len(result['sources'])}):\n")
for s in result["sources"]:
    print(f"  - {s['source_file']} (chunk {s['chunk_index']}, similarity: {s['score']:.3f})")

### 5.5 Inspect the RAG Process

You can inspect each step independently — search for chunks without sending to the LLM,
or view the full prompt that was sent.

In [ ]:
# Search only — view retrieved chunks without querying the LLM
question = "What are the main topics covered in the documents?"
chunks = search_similar_chunks(question, top_k=5)

print(f"Top {len(chunks)} chunks for: '{question}'\n")
for i, c in enumerate(chunks, 1):
    print(f"--- Chunk {i} (score: {c['score']:.3f}) ---")
    print(f"Source: {c['source_file']}, chunk_index: {c['chunk_index']}")
    print(c["text"][:500])
    print()

In [ ]:
# View the full prompt sent to the LLM
prompt = build_rag_prompt(question, chunks)
print(f"Prompt length: {len(prompt)} characters\n")
print(prompt)

In [ ]:
# Query the LLM directly (without RAG context) for comparison
direct_response = llm_client.chat.completions.create(
    model=LLM_SERVED_NAME,
    messages=[{"role": "user", "content": question}],
    max_tokens=512,
    temperature=0.1,
)

print("LLM response WITHOUT context (no RAG):\n")
print(direct_response.choices[0].message.content)
print("\n" + "=" * 60)
print("\nNotice how the answer without RAG context is generic,")
print("while the RAG answer references specific document content.")

### 5.6 Interactive Query Loop

Run this cell to enter an interactive Q&A session.
Type your questions and get RAG-powered answers. Type `quit` to exit.

In [ ]:
while True:
    question = input("\nAsk a question (or 'quit'): ").strip()
    if question.lower() in ("quit", "exit", "q"):
        print("Goodbye!")
        break
    if not question:
        continue

    result = rag_query(question)
    print(f"\nAnswer:\n{result['answer']}\n")
    print(f"Sources:")
    for s in result["sources"]:
        print(f"  - {s['source_file']} (chunk {s['chunk_index']}, {s['score']:.3f})")

---
## 6. Cleanup

Uncomment and run the cells below to tear down resources.

In [ ]:
# Cleanup resources using Kubernetes client
from kubernetes import client
from kubernetes.client.rest import ApiException

custom_api = client.CustomObjectsApi()
v1 = client.CoreV1Api()
rbac_v1 = client.RbacAuthorizationV1Api()

# Uncomment the sections you want to execute:

# Delete the LLM InferenceService (stops the model serving pod)
# try:
#     custom_api.delete_namespaced_custom_object(
#         group="serving.kserve.io",
#         version="v1beta1",
#         namespace=NAMESPACE,
#         plural="inferenceservices",
#         name=LLM_SERVED_NAME
#     )
#     print(f"✅ Deleted InferenceService '{LLM_SERVED_NAME}'")
# except ApiException as e:
#     print(f"Error deleting InferenceService: {e}")

# Delete the LLM ServingRuntime
# try:
#     custom_api.delete_namespaced_custom_object(
#         group="serving.kserve.io",
#         version="v1alpha1",
#         namespace=NAMESPACE,
#         plural="servingruntimes",
#         name=LLM_SERVED_NAME
#     )
#     print(f"✅ Deleted ServingRuntime '{LLM_SERVED_NAME}'")
# except ApiException as e:
#     print(f"Error deleting ServingRuntime: {e}")

# Delete the Embedding InferenceService (if using service mode)
# if EMBEDDING_MODE == "service":
#     try:
#         custom_api.delete_namespaced_custom_object(
#             group="serving.kserve.io",
#             version="v1beta1",
#             namespace=NAMESPACE,
#             plural="inferenceservices",
#             name=EMBEDDING_SERVICE_NAME
#         )
#         print(f"✅ Deleted InferenceService '{EMBEDDING_SERVICE_NAME}'")
#     except ApiException as e:
#         print(f"Error deleting embedding InferenceService: {e}")
#     
#     try:
#         custom_api.delete_namespaced_custom_object(
#             group="serving.kserve.io",
#             version="v1alpha1",
#             namespace=NAMESPACE,
#             plural="servingruntimes",
#             name=EMBEDDING_SERVICE_NAME
#         )
#         print(f"✅ Deleted ServingRuntime '{EMBEDDING_SERVICE_NAME}'")
#     except ApiException as e:
#         print(f"Error deleting embedding ServingRuntime: {e}")

# Drop the Milvus collection
# milvus_client.drop_collection(COLLECTION_NAME)
# print(f"✅ Dropped collection: {COLLECTION_NAME}")

# Delete the model cache PVC (removes downloaded model weights)
# try:
#     v1.delete_namespaced_persistent_volume_claim(
#         name=MODEL_CACHE_PVC,
#         namespace=NAMESPACE
#     )
#     print(f"✅ Deleted PVC '{MODEL_CACHE_PVC}'")
# except ApiException as e:
#     print(f"Error deleting PVC: {e}")

# Delete RBAC
# try:
#     rbac_v1.delete_namespaced_role(name="pipeline-rag-role", namespace=NAMESPACE)
#     print("✅ Deleted Role 'pipeline-rag-role'")
# except ApiException as e:
#     print(f"Error deleting Role: {e}")
# 
# try:
#     rbac_v1.delete_namespaced_role_binding(name="pipeline-rag-rolebinding", namespace=NAMESPACE)
#     print("✅ Deleted RoleBinding 'pipeline-rag-rolebinding'")
# except ApiException as e:
#     print(f"Error deleting RoleBinding: {e}")

print("💡 Uncomment the sections above to execute cleanup operations")